# IBKR Setup — Zero to Live Options Data

This guide walks you through the complete setup from creating an account to fetching your first live option chain. Total time: ~15 minutes.

**What you'll have at the end:**
- A free IBKR paper trading account
- TWS running with API enabled
- OptiCore fetching live option chains with IV + Greeks

## Overview

```
[1] Create IBKR account (free)
        ↓
[2] Download & install TWS
        ↓
[3] Log in to paper trading
        ↓
[4] Enable API in TWS settings
        ↓
[5] oc.check_connection() → Connected!
        ↓
[6] oc.fetch_chain("AAPL") → Live data!
```

---

## Step 1: Create an IBKR Account (free)

You need an Interactive Brokers account. A **paper trading** account is completely free — no funding required, no real money involved. It gives you simulated market data and full API access.

### 1a. Sign up

1. Go to **[ibkr.com/register](https://www.interactivebrokers.com/en/trading/free-trial.php)**
2. Click **"Try for Free"** or **"Open Account"**
3. Choose **Individual** account type
4. Fill in your details:
   - Name, email, country of residence
   - You can use your real info — this is just for account creation
5. Create a **username** and **password** (save these — you'll need them to log in to TWS)
6. Complete the application:
   - For paper trading only, the financial questions don't matter much
   - Select **"United States"** as your base currency if trading US equities
   - For trading permissions, enable **"US Equity Options"**

### 1b. Activate paper trading

Once your account is created (may take a few hours for approval):

1. Log in at **[ibkr.com](https://www.interactivebrokers.com)**
2. Go to **Settings (gear icon) > Account Settings**
3. Under **Paper Trading Account**, click **"Create"** or it may already be active
4. Your paper trading username is usually your live username with **"DU"** prefix (e.g., `DU1234567`)
5. The password is the same as your main account

> **Note:** You can start with the paper trading account immediately — no need to fund the account or wait for full verification.

---

## Step 2: Download and Install TWS

**Trader Workstation (TWS)** is IBKR's desktop application. It includes the API server that OptiCore connects to.

### macOS

1. Go to **[ibkr.com/en/trading/tws.php](https://www.interactivebrokers.com/en/trading/tws.php)**
2. Under **"TWS Latest"**, click the **macOS** download button
3. Open the `.dmg` file and drag TWS to your Applications folder
4. On first launch, macOS may block it:
   - Go to **System Settings > Privacy & Security**
   - Click **"Open Anyway"** next to the TWS message

### Windows

1. Same download page — click the **Windows** button
2. Run the installer, accept defaults
3. TWS will be installed to `C:\Jts\`

### Linux

1. Same download page — click the **Linux** button
2. Run: `chmod +x tws-latest-standalone-linux-x64.sh && ./tws-latest-standalone-linux-x64.sh`

> **Alternative: IB Gateway** — If you prefer a lighter app with no trading GUI, download
> [IB Gateway](https://www.interactivebrokers.com/en/trading/ibgateway-stable.php) instead.
> It uses ports 4001 (live) / 4002 (paper) by default.

---

## Step 3: Log In and Enable the API

### 3a. Log in to TWS

1. Launch TWS
2. At the login screen:
   - Enter your **username** and **password**
   - Select **"Paper Trading"** (not "Live Trading") from the dropdown at the top
3. Click **Log In**
4. Wait for TWS to fully load (the main trading window appears)

### 3b. Enable API connections

This is the critical step — TWS must allow external programs (like OptiCore) to connect.

1. In TWS, go to **Edit > Global Configuration** (Mac: **TWS > Global Configuration**)
2. In the left sidebar, expand **API** and click **Settings**
3. Configure these settings:

| Setting | Value | Why |
|---------|-------|-----|
| **Enable ActiveX and Socket Clients** | **Checked** | This is the master switch — without it, no API connections work |
| **Read-Only API** | Checked (recommended) | Prevents accidental order placement via API |
| **Socket port** | `7497` (live) or `7496` (paper) | Port that OptiCore connects to |
| **Allow connections from localhost only** | Checked | Security — only your machine can connect |
| **Trusted IPs** | Add `127.0.0.1` | Allows connections without manual approval popups |

4. Click **Apply**, then **OK**

### 3c. Disable the "Accept incoming" popup (optional but recommended)

By default, TWS pops up a dialog every time a program connects. To disable:

1. Still in **API > Settings**
2. Uncheck **"Allow connections from localhost only"** temporarily
3. Add `127.0.0.1` to the **Trusted IPs** list
4. Re-check **"Allow connections from localhost only"**

This tells TWS to silently accept connections from your own machine.

### Port cheat sheet

| Application | Live | Paper |
|-------------|------|-------|
| **TWS**     | 7497 | 7496  |
| **IB Gateway** | 4001 | 4002 |

> **Important:** Use the **paper** port while developing. Paper trading = simulated market with no real money.

---

## Step 4: Install the Python dependency

Make sure `ib_async` is installed alongside OptiCore.

In [5]:
!pip install ib_async

---

## Step 5: Test the Connection

With TWS running and API enabled, run this cell. Change the port if you're using paper trading (7496) or IB Gateway (4001/4002).

In [9]:
import opticore as oc

# ── Change this to match your setup ──
PORT = 7497   # TWS live=7497, paper=7496 | Gateway live=4001, paper=4002

status = oc.check_connection(port=PORT)
print(status["message"])

if status["connected"]:
    print(f"\n  Account:  {status['account']}")
    print(f"  Server:   v{status['server_version']}")
    print(f"\n  You're ready! Continue to Step 6.")

Connected to IBKR at 127.0.0.1:7497 (account: DUP812436, server v178)

  Account:  DUP812436
  Server:   v178

  You're ready! Continue to Step 6.


---

## Step 6: Fetch Your First Live Option Chain

One line to fetch, one line to enrich with IV + Greeks. OptiCore handles the connection lifecycle automatically — connects, fetches, disconnects.

In [10]:
# Fetch option chain — try AAPL, SPY, MSFT, or any optionable US equity
chain = oc.fetch_chain("AAPL", port=PORT, max_expiries=4, strike_count=15)
chain.head(10)

open orders request timed out
completed orders request timed out
Error 10089, reqId 4: Requested market data requires additional subscription for API. See link in 'Market Data Connections' dialog for more details.AAPL NASDAQ.NMS/TOP/ALL, contract: Stock(conId=265598, symbol='AAPL', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='AAPL', tradingClass='NMS')


ValueError: Could not get price for AAPL. Check your market data subscription.

In [ ]:
# Enrich with IV + Greeks
enriched = oc.enrich(chain, rate=0.045)
enriched[["strike", "expiry", "kind", "mid", "iv", "delta", "gamma", "theta", "vega"]].head(10)

In [ ]:
# Plot the IV smile from live market data
fig = oc.plot.smile(enriched)

---

## Quick Reference

```python
import opticore as oc

# 1. Test connection
status = oc.check_connection(port=7497)

# 2. Fetch chain (auto connects + disconnects)
chain = oc.fetch_chain("SPY", port=7497)

# 3. Enrich with IV + Greeks
enriched = oc.enrich(chain, rate=0.045, div_yield=0.013)

# 4. Plot
oc.plot.smile(enriched)                    # IV smile
oc.plot.smile(enriched, x="moneyness")     # smile vs moneyness
oc.plot.smile(enriched, expiry="20260515")  # single expiry
```

### `fetch_chain()` parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `symbol` | required | Ticker: "AAPL", "SPY", "TSLA", etc. |
| `port` | 7497 | TWS/Gateway port |
| `max_expiries` | 6 | Number of nearest expiry dates to fetch |
| `strike_count` | 20 | Strikes on each side of ATM |
| `market_data_type` | 3 | 1=live (paid), **3=delayed (free)**, 4=frozen |

### `enrich()` parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `chain` | required | DataFrame from `fetch_chain()` |
| `rate` | 0.045 | Risk-free rate |
| `div_yield` | 0.0 | Continuous dividend yield |
| `price_col` | "mid" | Which price to use: "mid", "bid", "ask", "last" |

---

## Troubleshooting

### Connection issues

| Error | Cause | Fix |
|-------|-------|-----|
| `Connection refused` | TWS/Gateway not running | Launch TWS and log in |
| `Connection refused` | API sockets not enabled | Edit > Global Configuration > API > Settings > enable sockets |
| `timed out` | Wrong port number | Check port: TWS paper=7496, live=7497 |
| `timed out` | Firewall blocking | Allow TWS through your system firewall |
| `ib_async not installed` | Missing dependency | `pip install ib_async` |

### Data issues

| Error | Cause | Fix |
|-------|-------|-----|
| `Could not get price for XXXX` | No market data subscription | Use `market_data_type=3` (delayed, free) |
| `No valid option contracts` | Symbol has no options | Try a major ticker: AAPL, SPY, MSFT |
| All IV values are NaN | Prices are zero (market closed) | Try during market hours (9:30-16:00 ET) |
| Prices show as -1 | No data subscription for exchange | Delayed data (type 3) is free for US equities |

### Market data types explained

| Type | Cost | Delay | When to use |
|------|------|-------|-------------|
| 1 | Paid subscription | Real-time | Production / live trading |
| **3** | **Free** | **15 min** | **Development and testing** |
| 4 | Free | Last available | After hours / weekends |

> **Tip:** OptiCore defaults to `market_data_type=3` (delayed). This is free and sufficient for
> development. You'll see `DELAYED` prefix on data in TWS — that's expected.

### Still stuck?

1. Make sure TWS is **fully loaded** (not just the login screen)
2. Try restarting TWS after changing API settings
3. Check if another program is using the same `client_id` (default: 1)
   - Change with: `oc.fetch_chain("AAPL", client_id=2)`
4. On macOS, check **System Settings > Privacy & Security > Firewall** isn't blocking TWS